In [ ]:
import pandas as pd
import numpy as np
from pipeline.ingestion import load_data
from pipeline.features import build_features, FEATURE_COLS
from pipeline.score import isofrst_score_transaction, score_in_chunks
from pipeline.classification import analyze_threshold, classify_transactions
from pipeline.explaination import run_explanation
from pipeline.evaluation import evaluate_pipeline

# Feature Engineering

In [52]:
df = load_data("data/transactions_with_labels.csv")

# combine separate date/time fields into single datetime type
if "DateTime" not in df.columns:
    df["DateTime"] = pd.to_datetime(df["Date"] + " " + df["Time"])

# features are already present in the saved CSV — only build if missing
account_feature_cols = ["sender_fan_out_ratio", "sender_amount_cv", "receiver_fan_in_ratio"]
if not all(col in df.columns for col in account_feature_cols):
    from pipeline.explanation import prepare_attributes
    df = build_features(df)
    df = prepare_attributes(df)

print(f"Shape: {df.shape}")
print(df.columns.tolist())

Shape: (9504852, 37)
['Time', 'Date', 'Sender_account', 'Receiver_account', 'Amount', 'Payment_currency', 'Received_currency', 'Sender_bank_location', 'Receiver_bank_location', 'Payment_type', 'Is_laundering', 'Laundering_type', 'DateTime', 'log_amount', 'is_currency_conversion', 'sender_blacklist', 'sender_greylist', 'receiver_blacklist', 'receiver_greylist', 'sender_high_risk', 'receiver_high_risk', 'any_high_risk', 'just_below_2k', 'below_2k_margin', 'just_below_5k', 'below_5k_margin', 'just_below_10k', 'below_10k_margin', 'just_below_50k', 'below_50k_margin', 'is_off_hours', 'is_weekend', 'sender_fan_out_ratio', 'sender_amount_cv', 'sender_unique_receiver_countries', 'receiver_fan_in_ratio', 'time_since_last_tx_hours']


# Getting Rid of Laundering Columns

In [53]:
df_with_labels = new_df.copy()

df_without_labels = new_df.drop(
    columns=["Is_laundering", "Laundering_type"],
    errors="ignore"
).copy()

print("With labels:", df_with_labels.shape)
print("Without labels:", df_without_labels.shape)
# df_with_labels.to_csv("data/transactions_with_labels.csv", index=False)
# df_without_labels.to_csv("data/transactions_without_labels.csv", index=False)

With labels: (9504852, 41)
Without labels: (9504852, 39)


In [ ]:
# Scoring Transactions with Isolation Forest
SCORING_FEATURES = [
    "log_amount",
    "sender_fan_out_ratio",
    "sender_amount_cv",
    "receiver_fan_in_ratio",
    "time_since_last_tx_hours",
    "is_currency_conversion",
    "any_high_risk",
    "just_below_10k",
    "below_10k_margin",
    "sender_unique_receiver_countries",
]

scored_df, clf, scaler = isofrst_score_transaction(df_without_labels, features=SCORING_FEATURES)

print("\nScoring Results:")
print(f"Shape: {scored_df.shape}")
print(f"Anomaly Score Range: [{scored_df['anomaly_score'].min():.4f}, {scored_df['anomaly_score'].max():.4f}]")
print(f"Anomaly Score (normalized) Range: [{scored_df['anomaly_score_normalized'].min():.4f}, {scored_df['anomaly_score_normalized'].max():.4f}]")
print("\nTop 10 Highest Risk Transactions:")
print(scored_df.nlargest(10, 'anomaly_score')[['Sender_account', 'Receiver_account', 'Amount', 'anomaly_score', 'anomaly_score_normalized']])

KeyError: "['sender_fan_out_ratio', 'sender_amount_cv', 'receiver_fan_in_ratio', 'sender_unique_receiver_countries'] not in index"

# Evaluation & Classification

In [ ]:
# Merge scored results with ground truth labels for evaluation
evaluation_df = scored_df.merge(
    df_with_labels[["Sender_account", "Receiver_account", "DateTime", "Is_laundering"]],
    on=["Sender_account", "Receiver_account", "DateTime"],
    how="left"
)

print(f"Evaluation dataset shape: {evaluation_df.shape}")
print(f"Ground truth labeling completeness: {evaluation_df['Is_laundering'].notna().sum()} / {len(evaluation_df)}")

# Analyze performance at different thresholds
print("\n" + "="*60)
print("THRESHOLD ANALYSIS")
print("="*60)
threshold_results = analyze_threshold(
    evaluation_df,
    score_col="anomaly_score",
    label_col="Is_laundering",
    thresholds=[0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
)

Evaluation dataset shape: (9505522, 38)
Ground truth labeling completeness: 9505522 / 9505522

THRESHOLD ANALYSIS
Top 0.1% flagged: 9,645 transactions
  True positives: 336 / 9,873 (3.4% recall)
  Precision: 3.48%
  F1: 0.0344

Top 0.5% flagged: 47,528 transactions
  True positives: 1,873 / 9,873 (19.0% recall)
  Precision: 3.94%
  F1: 0.0653

Top 1.0% flagged: 95,056 transactions
  True positives: 3,092 / 9,873 (31.3% recall)
  Precision: 3.25%
  F1: 0.0589

Top 2.0% flagged: 190,111 transactions
  True positives: 4,433 / 9,873 (44.9% recall)
  Precision: 2.33%
  F1: 0.0443

Top 5.0% flagged: 475,277 transactions
  True positives: 6,079 / 9,873 (61.6% recall)
  Precision: 1.28%
  F1: 0.0251

Top 10.0% flagged: 950,553 transactions
  True positives: 7,029 / 9,873 (71.2% recall)
  Precision: 0.74%
  F1: 0.0146

ROC-AUC: 0.8904


In [ ]:
# Classify transactions using optimal threshold (1% flagged)
classified_df, cutoff = classify_transactions(
    scored_df,
    score_col="anomaly_score",
    threshold_pct=1.0
)

print(f"Classification threshold: {cutoff:.4f}")
print(f"\nTop anomalous transactions (sample):")
print(classified_df[classified_df["is_anomalous"] == 1].nlargest(5, "anomaly_score")[
    ["Sender_account", "Receiver_account", "Amount", "anomaly_score", "is_anomalous"]
])

Classification at top 1.0%: 95,050 flagged as anomalous (1.000% of total)
Classification threshold: 0.6167

Top anomalous transactions (sample):
        Sender_account Receiver_account         Amount  anomaly_score  \
5746945     9401291265       1614632149    9849.139648       0.780476   
8349457     4735340495       3366423085    9643.599609       0.777120   
7281101     5732243739       4032397938  104290.851562       0.774215   
6666104     3588788631       9417119982    9049.549805       0.774049   
8321966     7094124432       3818650181  170955.406250       0.773474   

         is_anomalous  
5746945             1  
8349457             1  
7281101             1  
6666104             1  
8321966             1  


# Explanation: Discovering AML Patterns

In [55]:
# Use df_with_labels which has both anomaly scores and ground truth
explanation_df = df_with_labels.merge(
    classified_df[["Sender_account", "Receiver_account", "DateTime", "is_anomalous"]],
    on=["Sender_account", "Receiver_account", "DateTime"],
    how="left"
)

# Generate alerts ranking suspicious patterns by risk ratio
alerts = run_explanation(
    explanation_df,
    min_support=0.001,
    min_risk_ratio=3.0,
    max_combination_size=3,
    top_n=50
)
print('\n')
print("Top 20 suspicious pattern via Risk Ratios")
print(alerts.head(20).to_string())

Explanation stage: 95,050 outliers vs 9,505,522 total
Single-attribute candidates: 31
Encoding outlier transactions for FP-Growth:
Running FP-Growth on 33,448 outlier transactions, 31 candidate predicates
Found 170 frequent itemsets.


Top 20 suspicious pattern via Risk Ratios
                                                                                            predicates  outlier_count  outlier_support  pop_rate  risk_ratio  n_attributes
rank                                                                                                                                                                      
1                                             Payment_type=Cross-border ∧ Sender_bank_location=Morocco           2404         0.025293  0.001937   13.057247             2
2                                        Sender_bank_location=Morocco ∧ is_currency_conversion_str=yes           2398         0.025233  0.001933   13.051187             2
3            Payment_type=Cross-border

# Pipeline Evaluation Report

In [61]:
# Comprehensive evaluation of the full pipeline
evaluation_df = classified_df.merge(                      
    df_with_labels[["Sender_account", "Receiver_account", "DateTime","Is_laundering", "Laundering_type"]],
    on=["Sender_account", "Receiver_account", "DateTime"],
    how="left"
)

evaluate_pipeline(
    df=evaluation_df,
    alerts=alerts,
    threshold_results=threshold_results,
    label_col="Is_laundering",
    laundering_type_col="Laundering_type"
)

Pipeline Evaluation Report


Dataset: 9,505,522 transactions, 9,873 labeled suspicious (0.104%)

 Precision / Recall Trade-off
   Threshold    Flagged       TP  Precision   Recall       F1
        0.1%      9,645      336      3.48%     3.4%   0.0344
        0.5%     47,528    1,873      3.94%    19.0%   0.0653
        1.0%     95,056    3,092      3.25%    31.3%   0.0589
        2.0%    190,111    4,433      2.33%    44.9%   0.0443
        5.0%    475,277    6,079      1.28%    61.6%   0.0251
       10.0%    950,553    7,029      0.74%    71.2%   0.0146

--- Explanation Alignment with Known AML Patterns ---

Top 20 alerts by risk ratio:
                                                                                            predicates  outlier_support  risk_ratio  outlier_count
rank                                                                                                                                              
1                                             Payment_type=C